# Predikcija (aproksimacija) cena nekretnina
**Segmenti projekta:**

1. Arhitektura i konfiguracija
2. Scraping i sirovi dataset
3. Validacija i preprocessing
4. Feature engineering
5. Eksplorativnaanaliza podataka
6. Trening i evaluacija modela
7. Izbor i čuvanje najboljeg modela
8. Predikcija za novi unos
9. Streamlit integracija


## 1. Arhitektura sistema

Kompletan tok podataka izgleda ovako:

```text
nekretnine.rs
      |
      v
scraper + parser
      |
      v
data/raw/dataset.csv
      |
      v
validacija + preprocessing + feature engineering
      |
      v
data/processed/processed_dataset.csv
      |
      v
trening 3 regresiona modela + evaluacija
      |
      v
models/best_model.pkl
      |
      v
Streamlit aplikacija 
```



In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path) -> Path:
    """Pronađi root projekta nezavisno od foldera iz kog je notebook pokrenut."""
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "raw" / "dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Root projekta nije pronađen.")

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed_dataset.csv"
TRAINING_PROJECT = PROJECT_ROOT / "steps" / "03_ml_training"
APP_PROJECT = PROJECT_ROOT / "steps" / "04_streamlit_app"
MODEL_PATH = TRAINING_PROJECT / "models" / "best_model.pkl"
COMPARISON_PATH = TRAINING_PROJECT / "models" / "model_comparison.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dataset: {RAW_DATA_PATH.exists()}")
print(f"Processed dataset: {PROCESSED_DATA_PATH.exists()}")
print(f"Best model: {MODEL_PATH.exists()}")

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Segment: scraping i parsiranje



- `PropertyListing` predstavlja jedan oglas
- `NekretnineRsScraper` implementira logiku specificnu za nekretnine.rs
- `parser.py` pretvara HTML/JSON u strukturisane podatke
- `save_data.py` uklanja duplikate i cuva CSV


In [ ]:
scraper_files = [
    PROJECT_ROOT / "scraper" / "scraper.py",
    PROJECT_ROOT / "scraper" / "parser.py",
    PROJECT_ROOT / "scraper" / "models.py",
    PROJECT_ROOT / "scraper" / "save_data.py",
    PROJECT_ROOT / "scraper" / "config.py",
]
pd.DataFrame({
    "fajl": [path.name for path in scraper_files],
    "postoji": [path.exists() for path in scraper_files],
    "veličina_bajta": [path.stat().st_size if path.exists() else 0 for path in scraper_files],
})

### Mehanizmi scrapera

- User-Agent header
- timeout za svaki zahtev
- retry za privremene HTTP greske
- delay izmedju zahteva
- validacija cene i kvadrature
- deduplikacija po URL-u
- logging toka rada i gresaka

Ova struktura omogucava dodavanje drugog sajta kreiranjem nove klase koja nasledjuje `BaseScraper`.

## 3. Sirovi (raw lol) dataset

Ucitavamo raw dataset iz scrapera i proveravamo oblik, tipove i kvalitet podataka

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
print(f"Broj redova: {len(raw_df):,}")
print(f"Broj kolona: {len(raw_df.columns)}")
display(raw_df.head())

In [ ]:
raw_overview = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "missing_%": raw_df.isna().mean().mul(100),
    "unique": raw_df.nunique(dropna=True),
})
display(raw_overview)

In [ ]:
critical_quality = {
    "invalid_price": int((raw_df["price"].isna() | (raw_df["price"] <= 0)).sum()),
    "invalid_area": int((raw_df["area"].isna() | (raw_df["area"] <= 0)).sum()),
    "invalid_rooms": int((raw_df["rooms"].isna() | (raw_df["rooms"] <= 0)).sum()),
    "duplicate_urls": int(raw_df.duplicated(subset=["url"]).sum()),
}
pd.Series(critical_quality, name="broj_redova").to_frame()

## 4. Segment: validacija i preprocessing

Kriticni atributi su `price`, `area` i `rooms`. Red bez tih podataka nije pogodan za treniranje

Pipeline redom radi:

1. proveru obaveznih kolona
2. uklanjanje nevalidnih kriticnih vrednosti
3. uklanjanje duplikata
4. obradu nedostajucih vrednosti
5. konverziju numerickih tipova
6. ciscenje tekstualnih kolona
7. uklanjanje outliera IQR metodom
8. feature engineering i čuvanje rezultata

In [ ]:
def iqr_bounds(series: pd.Series) -> tuple[float, float]:
    """Vrati donju i gornju IQR granicu."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_rows = pd.Series(False, index=raw_df.index)
outlier_details = []
for column in ["price", "area"]:
    lower, upper = iqr_bounds(raw_df[column].dropna())
    column_outliers = ~raw_df[column].between(lower, upper)
    outlier_rows |= column_outliers
    outlier_details.append({
        "column": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(column_outliers.sum()),
    })
display(pd.DataFrame(outlier_details))
print(f"Redovi označeni kao outlier u bar jednoj koloni: {int(outlier_rows.sum())}")

### Zasto IQR?

IQR ne zahteva normalnu distribuciju. To je korisno kod nekretnina, gde raspodela cena ima dugacak desni tail i veoma skupe ekstremne oglase. Granice su:

```text
donja granica = Q1 - 1.5 * IQR
gornja granica = Q3 + 1.5 * IQR
IQR = Q3 - Q1
```

In [ ]:
processed_df = pd.read_csv(PROCESSED_DATA_PATH)
pipeline_summary = pd.DataFrame({
    "dataset": ["raw", "processed"],
    "rows": [len(raw_df), len(processed_df)],
    "columns": [len(raw_df.columns), len(processed_df.columns)],
    "missing_values": [int(raw_df.isna().sum().sum()), int(processed_df.isna().sum().sum())],
})
display(pipeline_summary)
display(processed_df.head())

## 5. feature engineering

Novi features omogucavaju modelu da lakse uci odnose koji nisu direktno zapisani u sirovim kolonama:

- `building_age = tekuća godina - building_year`
- `floor_ratio = floor / total_floors`
- `elevator`, `terrace`, `parking` postaju 0/1
- `city` i `municipality` koriste one-hot encoding
- `price_per_m2 = price / area` služi za analizu, ali se **ne koristi za trening cene**



In [ ]:
engineered_columns = [
    column for column in processed_df.columns
    if column in {"price_per_m2", "building_age", "floor_ratio"}
    or column.startswith("city_")
    or column.startswith("municipality_")
]
print(f"Broj engineered kolona: {len(engineered_columns)}")
print(engineered_columns)

## 6. Segment: eksplorativna analiza podataka (Exploative Data Analysis)

EDA pomaze da razumemo raspodele, odnose izmedju promenljivih, nedostajuce vrednosti i potencijalne probleme pre treniranja

In [ ]:
display(processed_df.describe().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(processed_df["price"], bins=35, kde=True, ax=axes[0], color="#2563eb")
axes[0].set_title("Distribucija cena")
axes[0].set_xlabel("Cena (€)")

sns.boxplot(x=processed_df["price"], ax=axes[1], color="#14b8a6")
axes[1].set_title("Box plot cene nakon preprocessinga")
axes[1].set_xlabel("Cena (€)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=processed_df,
    x="area",
    y="price",
    hue="rooms",
    palette="viridis",
    alpha=0.7,
)
plt.title("Odnos kvadrature i cene")
plt.xlabel("Kvadratura (m²)")
plt.ylabel("Cena (€)")
plt.show()

In [ ]:
core_columns = [
    "price", "area", "rooms", "floor", "total_floors",
    "building_year", "elevator", "terrace", "parking",
    "building_age", "floor_ratio",
]
correlation = processed_df[core_columns].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Korelacija glavnih numeričkih feature-a")
plt.show()

In [ ]:
price_correlations = (
    processed_df.corr(numeric_only=True)["price"]
    .drop(labels=["price", "price_per_m2"], errors="ignore")
    .sort_values(key=abs, ascending=False)
)
display(price_correlations.head(15).to_frame("correlation_with_price"))

In [ ]:
municipality_columns = [
    column for column in processed_df.columns if column.startswith("municipality_")
]
municipality_analysis = []
for column in municipality_columns:
    mask = processed_df[column] == 1
    if mask.any():
        municipality_analysis.append({
            "municipality": column.replace("municipality_", ""),
            "listings": int(mask.sum()),
            "median_price": processed_df.loc[mask, "price"].median(),
            "median_price_per_m2": processed_df.loc[mask, "price_per_m2"].median(),
        })
municipality_df = pd.DataFrame(municipality_analysis).sort_values(
    "median_price_per_m2", ascending=False
)
display(municipality_df)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=municipality_df,
    x="median_price_per_m2",
    y="municipality",
    color="#2563eb",
)
plt.title("Medijalna cena po m² po opštini")
plt.xlabel("Medijalna cena po m² (€)")
plt.ylabel("Opština")
plt.show()

## 7. Segment: priprema podataka za trening

Target je `price`. Sve ostale odgovarajuće numeričke kolone čine `X`, osim `price_per_m2`, koja se uklanja zbog data leakagea

Podela je 80/20 uz `random_state=42`, sto omogucava reproduktivnost rezultata

In [ ]:
TARGET_COLUMN = "price"
LEAKAGE_COLUMNS = ["price_per_m2"]

X = processed_df.drop(columns=[TARGET_COLUMN, *LEAKAGE_COLUMNS], errors="ignore")
X = X.select_dtypes(include=["number", "bool"]).copy()
X = X.fillna(X.median(numeric_only=True))
y = processed_df[TARGET_COLUMN].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"X shape: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target train: {y_train.shape}, Target test: {y_test.shape}")

## 8. Segment: trening tri modela

Modeli imaju različite uloge:

- **LinearRegression**: jednostavan i interpretabilan baseline.
- **RandomForestRegressor**: ansambl stabala, modeluje nelinearne odnose.
- **GradientBoostingRegressor**: sekvencijalno popravlja greske prethodnih stabala.

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),
}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Treniran: {model_name}")

## 9. Segment: evaluacija

Koristimo tri regresione metrike:

- **MAE**: prosecna apsolutna greska u evrima. Lako se tumaci.
- **RMSE**: jace kaznjava velike greske. Primarni kriterijum izbora modela.
- **R²**: deo varijanse cene koji model objasnjava; "vece je bolje".

In [ ]:
evaluation_rows = []
prediction_store = {}

for model_name, model in models.items():
    predictions = model.predict(X_test)
    prediction_store[model_name] = predictions
    evaluation_rows.append({
        "model": model_name,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
    })

results_df = pd.DataFrame(evaluation_rows).sort_values(
    ["rmse", "r2"], ascending=[True, False]
).reset_index(drop=True)
display(results_df.style.format({"mae": "{:,.2f}", "rmse": "{:,.2f}", "r2": "{:.4f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=results_df, x="rmse", y="model", ax=axes[0], color="#2563eb")
axes[0].set_title("RMSE poređenje (manje je bolje)")

sns.barplot(data=results_df, x="r2", y="model", ax=axes[1], color="#14b8a6")
axes[1].set_title("R² poređenje (veće je bolje)")
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
price_min = min(y_test.min(), *(pred.min() for pred in prediction_store.values()))
price_max = max(y_test.max(), *(pred.max() for pred in prediction_store.values()))

for axis, (model_name, predictions) in zip(axes, prediction_store.items()):
    axis.scatter(y_test, predictions, alpha=0.65)
    axis.plot([price_min, price_max], [price_min, price_max], "r--")
    axis.set_title(model_name)
    axis.set_xlabel("Stvarna cena")
    axis.set_ylabel("Predviđena cena")
plt.tight_layout()
plt.show()

## 10. Izbor najboljeg modela

Najbolji model je onaj sa najmanjim RMSE. Ako su RMSE rezultati unutar 1%, prednost dobija model sa većim R².

In [ ]:
best_rmse = results_df.iloc[0]["rmse"]
close_candidates = results_df[results_df["rmse"] <= best_rmse * 1.01]
best_row = close_candidates.sort_values(["r2", "rmse"], ascending=[False, True]).iloc[0]
best_model_name = best_row["model"]
best_model_from_notebook = models[best_model_name]

print(f"Najbolji model: {best_model_name}")
print(f"RMSE: {best_row['rmse']:,.2f} €")
print(f"R²: {best_row['r2']:.4f}")

In [ ]:
if COMPARISON_PATH.exists():
    production_comparison = pd.read_csv(COMPARISON_PATH)
    print("Metrike sačuvane produkcionim trening pipeline-om:")
    display(production_comparison)
else:
    print("model_comparison.csv nije pronađen.")

## 11. Feature importance

Modeli bazirani na stablima daju procenu relativne vaznosti featurea. 

In [ ]:
importance_model = models["GradientBoostingRegressor"]
importance_df = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": importance_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .head(15)
)
display(importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x="importance", y="feature", color="#2563eb")
plt.title("Top 15 feature-a - Gradient Boosting")
plt.show()

## 12. Ucitavanje produkcionog modela

Produkcioni model je vec sačuvan pomocu `joblib`. Taj model koji koristi Streamlit aplikacija.

In [ ]:
production_model = joblib.load(MODEL_PATH)
print(f"Tip modela: {type(production_model).__name__}")
print(f"Broj očekivanih feature-a: {production_model.n_features_in_}")
print("Feature kolone:")
print(list(production_model.feature_names_in_))

## 13. Predikcija za novi korisnički unos

Korisnicki unos mora da prođe iste transformacije kao trening podaci:

- numericke vrednosti postaju `float`/`int`
- boolean vrednosti postaju 0/1
- racuna se starost zgrade
- racuna se odnos sprata i ukupnog broja spratova
- grad i opstina se pretvaraju u one-hot kolone
- konacni DataFrame se poravnava sa `feature_names_in_` modela

In [ ]:
from datetime import date

def prepare_prediction_input(user_input: dict, model) -> pd.DataFrame:
    """Pretvori korisnički unos u format koji produkcioni model očekuje."""
    total_floors = max(int(user_input["total_floors"]), 1)
    floor = int(user_input["floor"])
    building_year = int(user_input["building_year"])

    base = {
        "area": float(user_input["area"]),
        "rooms": int(user_input["rooms"]),
        "floor": floor,
        "total_floors": total_floors,
        "building_year": building_year,
        "elevator": int(user_input["elevator"]),
        "terrace": int(user_input["terrace"]),
        "parking": int(user_input["parking"]),
        "building_age": max(date.today().year - building_year, 0),
        "floor_ratio": floor / total_floors,
        f"city_{user_input['city']}": 1,
        f"municipality_{user_input['municipality']}": 1,
    }

    aligned = pd.DataFrame(
        0.0, index=[0], columns=list(model.feature_names_in_)
    )
    for column, value in base.items():
        if column in aligned.columns:
            aligned.loc[0, column] = value
    return aligned

example_property = {
    "area": 55.0,
    "rooms": 2,
    "city": "Beograd",
    "municipality": "Novi Beograd",
    "floor": 2,
    "total_floors": 5,
    "building_year": 2005,
    "elevator": True,
    "terrace": True,
    "parking": False,
}

example_features = prepare_prediction_input(example_property, production_model)
display(example_features)

In [ ]:
estimated_price = max(float(production_model.predict(example_features)[0]), 0.0)
print(f"Procenjena cena: {estimated_price:,.0f} €")

## 14. Segment: Streamlit aplikacija

Korisnik unosi podatke, `predict.py` priprema identicne feature kolone, model racuna rezultat, a aplikacija prikazuje procenjenu cenu.

Aplikacija sadrzi:

- unos kvadrature, soba, lokacije i podataka o zgradi
- checkbox kontrole za lift, terasu i parking
- dugme `Predvidi cenu`
- rezultat
- sidebar sa informacijama i disclaimer
- grafikon odnosa kvadrature i cene

